<a href="https://colab.research.google.com/github/manoj-1229/open-data-intelligence-hub/blob/main/Manoj_Reusable_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

# ==========================================
# 1. MOCK DATA GENERATION (With Real Learning Patterns)
# ==========================================
np.random.seed(42)
data_size = 1000

# Generate realistic underlying features
age = np.random.choice([25, 35, 45, 55, np.nan], size=data_size)
monthly_charges = np.random.choice([30.0, 60.0, 90.0, 120.0, np.nan], size=data_size)
contract = np.random.choice(['Month-to-month', 'One year', 'Two year'], size=data_size)
payment = np.random.choice(['Electronic check', 'Mailed check', 'Bank transfer'], size=data_size)

# Create a clean mathematical rule so the model can actually learn:
# High monthly charges + Month-to-month contracts = High probability of Churn (1)
churn_probability = np.zeros(data_size)
for i in range(data_size):
    prob = 0.2  # Base risk
    if contract[i] == 'Month-to-month':
        prob += 0.4
    if monthly_charges[i] >= 90.0:
        prob += 0.3
    churn_probability[i] = min(0.9, prob)

# Assign binary target classes based on calculated probabilities
churn = np.random.binomial(1, churn_probability)

mock_data = {
    'Age': age,
    'MonthlyCharges': monthly_charges,
    'ContractType': contract,
    'PaymentMethod': payment,
    'Churn': churn
}

df = pd.DataFrame(mock_data)

# Separate Features (X) and Target (y)
X = df.drop(columns=['Churn'])
y = df['Churn']

# Split into Training and Testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ==========================================
# 2. DEFINING THE REUSABLE PIPELINE
# ==========================================

# Identify feature types automatically
numeric_features = ['Age', 'MonthlyCharges']
categorical_features = ['ContractType', 'PaymentMethod']

# Step A: Transform Numeric Features (Impute missing data -> Scale values)
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Step B: Transform Categorical Features (Impute missing data -> One-Hot Encode)
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Step C: Combine Transformers into a Column Processing Factory Line
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

# Step D: Assemble the Master Pipeline (Preprocessing -> ML Model Estimation)
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(random_state=42))
])

# ==========================================
# 3. TRAINING & EVALUATING THE PIPELINE
# ==========================================

print("⏳ Training the reusable pipeline...")
pipeline.fit(X_train, y_train)
print("✅ Training complete!\n")

# Make predictions on test data
y_pred = pipeline.predict(X_test)

# Display evaluation metrics
print("--- Model Performance Metrics ---")
print(f"Accuracy Score: {accuracy_score(y_test, y_pred):.4f}\n")
print("Classification Report:")
print(classification_report(y_test, y_pred))

# ==========================================
# 4. REUSING THE PIPELINE FOR NEW DATA
# ==========================================
# Demonstrating how easy it is to process new incoming customer records raw
new_customers = pd.DataFrame({
    'Age': [32.0, np.nan],                  # Test missing value handling
    'MonthlyCharges': [120.00, 30.00],      # High charges vs Low charges
    'ContractType': ['Month-to-month', 'Two year'],
    'PaymentMethod': ['Electronic check', 'Bank transfer']
})

print("🔮 Predicting churn risk for incoming new data...")
new_predictions = pipeline.predict(new_customers)
print(f"Predictions (0 = Stay, 1 = Churn): {new_predictions}")

⏳ Training the reusable pipeline...
✅ Training complete!

--- Model Performance Metrics ---
Accuracy Score: 0.7450

Classification Report:
              precision    recall  f1-score   support

           0       0.74      0.81      0.78       108
           1       0.75      0.66      0.71        92

    accuracy                           0.74       200
   macro avg       0.75      0.74      0.74       200
weighted avg       0.75      0.74      0.74       200

🔮 Predicting churn risk for incoming new data...
Predictions (0 = Stay, 1 = Churn): [1 0]


## Task Documentation: Reusable ML Pipeline with scikit-learn

### 1. Project Objective
The goal of this project is to build a structured, production-ready machine learning workflow using `scikit-learn` pipelines. Instead of preprocessing data manually through scattered blocks of code, this factory-like assembly line approach ensures consistency, eliminates data leakage, and allows immediate inference on raw, unseen customer data.

### 2. Core Components of the Implemented Pipeline
* **Numerical Processing Branch:** Uses `SimpleImputer(strategy='median')` to cleanly handle missing values based on training metrics, followed by `StandardScaler()` to bring features like `Age` and `MonthlyCharges` onto a uniform scale.
* **Categorical Processing Branch:** Implements `SimpleImputer(strategy='most_frequent')` for missing text attributes and applies `OneHotEncoder(handle_unknown='ignore')` to safely convert categorical columns into numerical dimensions without crashing on new labels.
* **ColumnTransformer Integration:** Combines individual feature processing branches into a single unified transformation blueprint.
* **Model Estimator:** Links the data preprocessing blueprint directly with a `LogisticRegression` classifier, establishing a single object (`pipeline`) that manages everything from raw variables to predictions.

### 3. Production Readiness & Benefits
* **Data Leakage Mitigation:** Statistics for imputation and scaling are explicitly calculated only on the training subset (`X_train`) and safely applied down the line to testing segments, maintaining strict statistical validation.
* **Instant Inference Capability:** The production pipeline natively processes completely raw incoming data dataframes (containing missing values and text strings) and returns definitive churn predictions (`0` or `1`) instantly without manual data-cleaning intervention.